In [ ]:
# Lecture 20 — Parquet vs CSV: size, speed, and type safety
# Spark 101 · Module 6 · Lecture 20
#
# Import this file into a Databricks workspace as a notebook.
# Databricks recognizes `# COMMAND ----------` as a cell boundary on import.
# In VS Code / PyCharm this stays a plain Python file — same source, two homes.

In [ ]:
print("Spark version:", spark.version)
# Databricks gives you `spark` for free — no SparkSession.builder needed.

In [ ]:
# --- Build a dataset big enough for compression to matter ------------------
from pyspark.sql.functions import col, expr

# 200k rows with a date, an int, and some repetitive text (compresses well).
df = (
    spark.range(0, 200_000)
    .withColumn("event_date", expr("date_add(to_date('2026-01-01'), cast(id % 90 as int))"))
    .withColumn("country", expr("element_at(array('ES','US','FI','FR'), cast(id % 4 as int) + 1)"))
    .withColumn("amount", (col("id") % 1000).cast("double"))
)
df.printSchema()
df.show(5)

base = "/tmp/spark101_l20"

In [ ]:
# --- Write the SAME data both ways -----------------------------------------
df.write.mode("overwrite").option("header", True).csv(f"file:{base}/as_csv")
df.write.mode("overwrite").parquet(f"file:{base}/as_parquet")
print("wrote both formats")

In [ ]:
# --- Size on disk ----------------------------------------------------------
import os

def dir_size(path):
    total = 0
    for root, _, files in os.walk(path):
        for fn in files:
            total += os.path.getsize(os.path.join(root, fn))
    return total

csv_bytes = dir_size(f"{base}/as_csv")
pq_bytes = dir_size(f"{base}/as_parquet")
print(f"CSV total:     {csv_bytes:,} bytes")
print(f"Parquet total: {pq_bytes:,} bytes")
print(f"Parquet is ~{csv_bytes / pq_bytes:.1f}x smaller")
# Read the ACTUAL ratio off screen — don't quote a fixed number.
# Typical real-world range is 5–10x; exact value depends on the data.

In [ ]:
# --- Types survive the round trip? -----------------------------------------
csv_back = spark.read.option("header", True).csv(f"file:{base}/as_csv")
pq_back = spark.read.parquet(f"file:{base}/as_parquet")

print("=== CSV read back (types LOST — all strings) ===")
csv_back.printSchema()

print("=== Parquet read back (types KEPT) ===")
pq_back.printSchema()
# CSV: event_date and amount come back as `string`.
# Parquet: they come back as `date` and `double`, exactly as written.

In [ ]:
# --- Column read: SELECT one column ---------------------------------------
# Parquet reads only the `amount` column's bytes; CSV must read every column.
pq_back.select("amount").summary("count", "min", "max").show()

In [ ]:
# Compression codecs: snappy (default) vs gzip — compare size on disk.
df.write.mode("overwrite").option("compression", "snappy").parquet(f"file:{base}/snappy")
df.write.mode("overwrite").option("compression", "gzip").parquet(f"file:{base}/gzip")
print("snappy:", dir_size(f"{base}/snappy"), "bytes")
print("gzip:  ", dir_size(f"{base}/gzip"), "bytes")

In [ ]:
# Predicate pushdown: a filter pushes INTO the Parquet scan.
pq_back.filter("amount > 500").explain()   # look for PushedFilters

In [ ]:
# Selective column read — Parquet reads only the columns you ask for.
pq_back.select("country", "amount").explain()   # ReadSchema lists 2 columns

In [ ]:
# Recap:
#   - Same data: Parquet is several times smaller than CSV.
#   - Parquet keeps types across the round trip; CSV returns strings.
#   - Default to Parquet for pipelines; use CSV only for human handoffs.
# End of Module 6 — Writing & Saving Data.